In [1]:
# import libraries
import os
import glob
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from functools import reduce
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.functions import array_to_vector, vector_to_array
from pyspark.sql.types import BinaryType, ArrayType, FloatType
import io
from pyspark.sql.functions import rand, explode, array, lit, col, array_repeat, \
lit, create_map, when, ceil, udf, window, pandas_udf
from pyspark.ml.linalg import Vectors, VectorUDT
import math

# sns.set_theme(style = 'whitegrid', palette = 'viridis')
# plt.rcParams['figure.dpi'] = 120

print('Done')

Matplotlib created a temporary cache directory at /scratch/mfelix1/job_49171570/matplotlib-rr49j0w4 because the default path (/home/jovyan/.cache/matplotlib) is not a writable directory; it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


Done


In [2]:
# Config the spark session
spark = SparkSession.builder \
    .appName("plant-disease") \
    .config("spark.driver.memory", "2g") \
    .config("spark.executor.memory", "32g") \
    .config("spark.executor.instances", 4) \
    .getOrCreate()

print(f"Spark UI: {spark.sparkContext.uiWebUrl}")

Spark UI: http://exp-1-42.expanse.sdsc.edu:4041


### Import the dataset (not needed after it's done once)

In [3]:
# pip install kagglehub

In [4]:
# Link to dataset: https://www.kaggle.com/datasets/samareshkumar/multipleplantdiseases
# Import the dataset. unnecessary once you've done it once.

#import kagglehub

# # Download latest version
#path = kagglehub.dataset_download("samareshkumar/multipleplantdiseases")

# print("Path to dataset files:", path)

In [5]:
data_path = "/home/mfelix1/.cache/kagglehub/datasets/samareshkumar/multipleplantdiseases/versions/1"
print(f'Data path exists: {os.path.exists(data_path)}')

Data path exists: True


### Load in and clean data
We will create a dataframe with file path, class label (taken from the folder name), plant species, and disease.

First we'll clean the data, split into train/validation/test sets, and label encode. Then we will format it so it can be used inside Spark models.

We found that there were high data imbalances during our EDA. We won't change anything about the data in the preprocessing, but instead tune the weights parameter in our models to handle the imbalances algorithmically.

The chili image folder names weren't formatted like the others. The other plants have three underscores in between the plant name and disease name, like Wheat___brown_rust. For the chili plants, their folder names only have two underscores, such as Chili__yellowish. We'll split based on the two underscores to make sure we extract Chili correctly for the "plant" column. This will lead to our "diseases" column having preceding "_" for the values associated with the other plants, but for our purposes, this is okay. For our binary classification, we only want to know whether a plant is diseased or healthy, not the specific disease that the plant has. We will create a new column, health, for this.

It also looks like the data uses four ways to classify a plant as healthy: healthy, Healthy, healthy_leaf, and fresh_leaf. We'll want to combine those into one healthy class.

In [6]:
# Define where to split the class label to get the plant species and disease
split = F.split("class_label", "__")

# Create a Spark dataframe
metadata = spark.read.format("binaryFile") \
    .option("recursiveFileLookup", "true") \
    .load(data_path) \
    .withColumn("file_path", F.col("path")) \
    .withColumn("class_label", F.element_at(F.split("path", "/"), -2)) \
    .withColumn("plant", split.getItem(0)) \
    .withColumn("disease", split.getItem(1)) \
    .select("file_path", "class_label", "plant", "disease")

# Create a cleaned column for disease that combines the 4 different healthy labels into one
# Create a cleaned column from the disease column called health, which only identifies healthy or diseased.

# Then drop the cols we don't need to save storage.
metadata = metadata.withColumn(
    "disease_clean",
    F.when(F.lower(F.col("disease")).rlike("healthy|fresh_leaf"), "healthy")
     .otherwise(F.col("disease"))
).withColumn("health", \
 F.when(F.lower(F.col("disease_clean"))== "healthy", "healthy").otherwise("diseased"))\
.select("file_path", "plant", "health").cache()

In [7]:
metadata.printSchema()

root
 |-- file_path: string (nullable = true)
 |-- plant: string (nullable = true)
 |-- health: string (nullable = false)



Verify that the plant species and diseases were extracted correctly, and that we have no NULL values for our labels.

In [8]:
metadata.select("health").distinct().show(40)

+--------+
|  health|
+--------+
| healthy|
|diseased|
+--------+



In [9]:
metadata.count()

52360

### Split into train, validation, and test sets
We will create two sets of train, test, and validation data. One will be for our binary classification of diseased vs healthy plants, and the other will be for our multiclass classification of plant species.

We will do this using stratified random sampling since we have rather large class imbalances to ensure that our smaller classes are still represented in the training data.

Model 2 - Multiclassification of plant species has been moved to end of the notebook

In [10]:
# We want 70% of healthy samples and 70% of diseased samples to go into our training data
# 15% of healthy/diseased samples should go into validation, and 15% into test sets.

w_d = Window.partitionBy("health").orderBy(F.rand(seed=1))

metadata_d = metadata.withColumn("row_num", F.row_number().over(w_d))\
            .withColumn("count", F.count("*").over(Window.partitionBy("health")))

metadata_d = metadata_d.withColumn("split", F.when(F.col("row_num") <= 0.7* F.col("count"), "train")\
            .when(F.col("row_num") <= 0.85 * F.col("count"), "val").otherwise("test"))

train_d = metadata_d.filter(F.col("split") == "train").select("file_path", "health")
val_d = metadata_d.filter(F.col("split") == "val").select("file_path", "health")
test_d = metadata_d.filter(F.col("split") == "test").select("file_path", "health")

In [11]:
train_d.printSchema()

root
 |-- file_path: string (nullable = true)
 |-- health: string (nullable = false)



In [12]:
# Check counts
print(train_d.count())
print(val_d.count())
print(test_d.count())

36651
7854
7855


### Label encode
Label encoding for binary (diseased vs healthy) classification

In [13]:
# Create label encoder and fit to the training set 
indexer_d = StringIndexer(inputCol="health", outputCol="label") 
le_d = indexer_d.fit(train_d)

# Label encode all sets
# 0.0 = diseased
# 1.0 = healthy
train_d = le_d.transform(train_d)
val_d = le_d.transform(val_d)
test_d = le_d.transform(test_d)

In [14]:
# Let's just check a few rows after label encoding
# print(train_d.take(3))
# print(val_d.take(3))
# print(test_d.take(3))
# print(train_d.count())

In [15]:
# Add a weights column for the weightcol parameter to tune the models because of our data imbalance
# ONLY do this for training data
train_d.cache()

class_counts_d = train_d.groupBy("label").count()

total_d = train_d.count()
num_classes_d = class_counts_d.count()

# set a max weight because without it, some classes are weighted like 8 and some 0.22 
# max_weight = 5.0

weights_d = class_counts_d.withColumn(
    "weight",
    F.lit(train_d.count()) / (F.lit(class_counts_d.count()) * F.col("count"))
).select("label", "weight")

# join the weights to the training data
train_d = train_d.join(weights_d, on="label", how="left")

In [16]:
# Check that the weights assigned correctly
train_d.groupBy("label").agg(F.first("weight")).show()

+-----+------------------+
|label|     first(weight)|
+-----+------------------+
|  0.0|0.6600453825097248|
|  1.0|2.0620569370991335|
+-----+------------------+



### Preprocess Images for Models

We must turn the images into numeric feature vectors in order for Spark models to be able to use them. The reason why I'm only bringing the images in now and not earlier in the preprocessing is because I was running into memory issues before and I wanted to be able to make sure I was splitting everything correctly.

In [17]:
# Bring in the images data now
images = spark.read.format("binaryFile").option("recursiveFileLookup", "true").load(data_path)\
            .select(F.col("path").alias("file_path"), F.col("content"))

In [18]:
# Join the images to the data
# Doing this now because it was giving me memory problems when I brought in the images earlier
# with all of the other transformations I did

train_d = train_d.join(images, on = "file_path", how = "inner")
val_d = val_d.join(images, on = "file_path", how = "inner")
test_d = test_d.join(images, on = "file_path", how = "inner")

In [19]:
train_d.printSchema()

root
 |-- file_path: string (nullable = true)
 |-- label: double (nullable = false)
 |-- health: string (nullable = false)
 |-- weight: double (nullable = true)
 |-- content: binary (nullable = true)



In [20]:
# UDF to process the Spark binary image into one that the ML models can use

@pandas_udf(ArrayType(FloatType()))
def decode_image(contents: pd.Series) -> pd.Series:
    result = []

    for content in contents:
        try:
            # read image from bytes
            img = Image.open(io.BytesIO(content)).convert("RGB")
            
            # resize, keep small bc of memory
            img = img.resize((32,32))
           
            # convert to np and normalize
            arr = np.asarray(img, dtype = np.float32) / 255.0

            # return Vector type for the spark models
            result.append(arr.reshape(-1).tolist())
        except Exception:
            result.append(None)

    return pd.Series(result)

In [21]:
train_d = train_d.withColumn("features_array", decode_image("content")) \
        .filter(F.col("features_array").isNotNull()) \
        .withColumn("features", array_to_vector("features_array"))\
        .select("file_path", "label", "weight", "features")

val_d = val_d.withColumn("features_array", decode_image("content")) \
        .filter(F.col("features_array").isNotNull()) \
        .withColumn("features", array_to_vector("features_array"))\
        .select("file_path", "label", "features")

test_d = test_d.withColumn("features_array", decode_image("content")) \
        .filter(F.col("features_array").isNotNull()) \
        .withColumn("features", array_to_vector("features_array"))\
        .select("file_path", "label", "features")

In [22]:
val_d.cache()

DataFrame[file_path: string, label: double, features: vector]

In [23]:
# check decoding
val_d.filter(F.col("features").isNotNull()).count()

7854

In [24]:
train_d.printSchema()

root
 |-- file_path: string (nullable = true)
 |-- label: double (nullable = false)
 |-- weight: double (nullable = true)
 |-- features: vector (nullable = true)



### MODEL #1 - XGBoost
Conducting a binary classification for diseases using XGBoost (distributed). Testing 3 different hyperparameter options with option 3 serving as a baseline for speedup analysis.

In [25]:
# import libraries for model
import xgboost
from xgboost.spark import SparkXGBClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
import time

In [26]:
# create evaluator
eval = BinaryClassificationEvaluator(labelCol='label', rawPredictionCol='probability')

#### Hyperparameter Option #1

Hyperparmeters: num_workers=4, num_rounds=100, eta=0.1

In [27]:
h1_xgb = SparkXGBClassifier(
    features_col="features",
    label_col="label",
    num_workers=4,
    num_rounds=100,
    eta=0.1,
    seed=42
)

h1_model = h1_xgb.fit(train_d)

# validation predictions & evaluation
h1_val_pred = h1_model.transform(val_d).cache()
h1_val_eval = eval.evaluate(h1_val_pred)

print("Hyperparmeter Option #1")
print("=" * 50)
print("Validation predictions & evaluation:")
print(f"AUC: {h1_val_eval}")
print(h1_val_pred.show(3))

2026-05-19 07:07:50,450 INFO XGBoost-PySpark: _fit Running xgboost-2.0.3 on 4 workers with
	booster params: {'objective': 'binary:logistic', 'device': 'cpu', 'num_rounds': 100, 'eta': 0.1, 'seed': 42, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 100}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
2026-05-19 07:17:38,590 INFO XGBoost-PySpark: _fit Finished xgboost training!


Hyperparmeter Option #1
Validation predictions & evaluation:
AUC: 0.9263584845703003
+--------------------+-----+--------------------+--------------------+----------+--------------------+
|           file_path|label|            features|       rawPrediction|prediction|         probability|
+--------------------+-----+--------------------+--------------------+----------+--------------------+
|file:/home/mfelix...|  1.0|[0.81176471710205...|[0.00723810726776...|       0.0|[0.50180947780609...|
|file:/home/mfelix...|  0.0|[0.50588238239288...|[2.18683481216430...|       0.0|[0.89906102418899...|
|file:/home/mfelix...|  0.0|[0.95294117927551...|[0.18679299950599...|       0.0|[0.54656291007995...|
+--------------------+-----+--------------------+--------------------+----------+--------------------+
only showing top 3 rows

None


#### Hyperparameter Option #2

Hyperparmeters: num_workers=4, num_rounds=50, eta=0.5

In [28]:
start = time.time() # track time
h2_xgb = SparkXGBClassifier(
    features_col="features",
    label_col="label",
    num_workers=4,
    num_rounds=50,  # num_rounds decease to 50
    eta=0.5,         # eta increased to 0.5
    seed=42
)

h2_model = h2_xgb.fit(train_d)

# validation predictions & evaluation
h2_val_pred = h2_model.transform(val_d).cache()
h2_val_eval = eval.evaluate(h2_val_pred)

T_4 = time.time() - start

print("Hyperparmeter Option #2")
print(f"Time: {T_4}")
print("=" * 50)
print("Validation predictions & evaluation:")
print(f"AUC: {h2_val_eval}")
print(h2_val_pred.show(3))

2026-05-19 07:22:59,664 INFO XGBoost-PySpark: _fit Running xgboost-2.0.3 on 4 workers with
	booster params: {'objective': 'binary:logistic', 'device': 'cpu', 'num_rounds': 50, 'eta': 0.5, 'seed': 42, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 100}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
2026-05-19 07:32:59,418 INFO XGBoost-PySpark: _fit Finished xgboost training!


Hyperparmeter Option #2
Time: 916.5659439563751
Validation predictions & evaluation:
AUC: 0.9380689481675041
+--------------------+-----+--------------------+--------------------+----------+--------------------+
|           file_path|label|            features|       rawPrediction|prediction|         probability|
+--------------------+-----+--------------------+--------------------+----------+--------------------+
|file:/home/mfelix...|  1.0|[0.81176471710205...|[-1.7572298049926...|       1.0|[0.14713758230209...|
|file:/home/mfelix...|  0.0|[0.50588238239288...|[5.04311370849609...|       0.0|[0.99358773231506...|
|file:/home/mfelix...|  0.0|[0.95294117927551...|[2.00626301765441...|       0.0|[0.88145309686660...|
+--------------------+-----+--------------------+--------------------+----------+--------------------+
only showing top 3 rows

None


#### Hyperparameter Option #3 - Baseline
Create baseline keep model same as hyperparameter option #2 with exception of decrease in num_workers

Hyperparmeters: num_workers=1, num_rounds=50, eta=0.5

In [29]:
start = time.time()
h3_xgb = SparkXGBClassifier(
    features_col="features",
    label_col="label",
    num_workers=1,   # decrease num_worker to 1
    num_rounds=50,
    eta=0.5,
    seed=42
)

h3_model = h3_xgb.fit(train_d)

# validation predictions & evaluation
h3_val_pred = h3_model.transform(val_d).cache()
h3_val_eval = eval.evaluate(h3_val_pred)

T_B = time.time() - start 

print("Hyperparmeter Option #3")
print(f"Time: {T_B}")
print("=" * 50)
print("Validation predictions & evaluation:")
print(f"AUC: {h3_val_eval}")
print(h3_val_pred.show(3))

2026-05-19 07:38:17,025 INFO XGBoost-PySpark: _fit Running xgboost-2.0.3 on 1 workers with
	booster params: {'objective': 'binary:logistic', 'device': 'cpu', 'num_rounds': 50, 'eta': 0.5, 'seed': 42, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 100}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
2026-05-19 07:47:49,579 INFO XGBoost-PySpark: _fit Finished xgboost training!


Hyperparmeter Option #3
Time: 835.3542232513428
Validation predictions & evaluation:
AUC: 0.9380689481675041
+--------------------+-----+--------------------+--------------------+----------+--------------------+
|           file_path|label|            features|       rawPrediction|prediction|         probability|
+--------------------+-----+--------------------+--------------------+----------+--------------------+
|file:/home/mfelix...|  1.0|[0.81176471710205...|[-1.7572298049926...|       1.0|[0.14713758230209...|
|file:/home/mfelix...|  0.0|[0.50588238239288...|[5.04311370849609...|       0.0|[0.99358773231506...|
|file:/home/mfelix...|  0.0|[0.95294117927551...|[2.00626301765441...|       0.0|[0.88145309686660...|
+--------------------+-----+--------------------+--------------------+----------+--------------------+
only showing top 3 rows

None


In [31]:
h1_val_pred.unpersist()
h2_val_pred.unpersist()
h3_val_pred.unpersist()

DataFrame[file_path: string, label: double, features: vector, rawPrediction: vector, prediction: double, probability: vector]

#### Final predictions
Perform & evaluate best perform model on test data. Chose option 2 due to higher AUC. Even though option 2 & option 3 had same AUC, went with option 2 because options 3 was only created to serve as baseline for speed up.

##### Train prediction & evaluation

In [31]:
# Make prediction using best model
best_xgb = SparkXGBClassifier(
    features_col="features",
    label_col="label",
    num_workers=4,
    num_rounds=50,  # num_rounds decease to 50
    eta=0.5,         # eta increased to 0.5
    seed=42
)

best_model = best_xgb.fit(train_d)

best_train_pred = best_model.transform(train_d)
best_train_eval = eval.evaluate(best_train_pred)

test_pred = best_model.transform(test_d)
test_eval = eval.evaluate(test_pred)

print("Train predictions & evaluation:")
print("=" * 50)
print(f"AUC: {best_train_eval}")
print(best_train_pred.show(3))

print("Test predictions & evaluation:")
print("=" * 50)
print(f"AUC: {test_eval}")
print(test_pred.show(3))

2026-05-19 10:00:43,505 INFO XGBoost-PySpark: _fit Running xgboost-2.0.3 on 4 workers with
	booster params: {'objective': 'binary:logistic', 'device': 'cpu', 'num_rounds': 50, 'eta': 0.5, 'seed': 42, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 100}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
2026-05-19 10:11:27,025 INFO XGBoost-PySpark: _fit Finished xgboost training!


Train predictions & evaluation:
AUC: 0.9999996413209137
+--------------------+-----+------------------+--------------------+--------------------+----------+--------------------+
|           file_path|label|            weight|            features|       rawPrediction|prediction|         probability|
+--------------------+-----+------------------+--------------------+--------------------+----------+--------------------+
|file:/home/mfelix...|  1.0|2.0620569370991335|[0.64705884456634...|[-1.8559296131134...|       1.0|[0.13517820835113...|
|file:/home/mfelix...|  1.0|2.0620569370991335|[0.81176471710205...|[-1.7572298049926...|       1.0|[0.14713758230209...|
|file:/home/mfelix...|  1.0|2.0620569370991335|[0.52156865596771...|[-3.2233555316925...|       1.0|[0.03829628229141...|
+--------------------+-----+------------------+--------------------+--------------------+----------+--------------------+
only showing top 3 rows

None
Test predictions & evaluation:
AUC: 0.9388896534991906
+----

### Speedup Table
Conducted on Options 2 & 3 of Distributed XGBoost Model.

In [36]:
def create_speedup_table(measurements):
    T_1 = measurements[0][1]  # Baseline time

    data = []
    for n_workers, exec_time in measurements:
        speedup = T_1 / exec_time
        efficiency = speedup / n_workers

        data.append({
            'Executors': n_workers,
            'Time (sec)': f'{exec_time:.1f}',
            'Speedup': f'{speedup:.2f}x',
            'Efficiency': f'{efficiency:.0%}'
        })

    return pd.DataFrame(data)

example_measurements = [
    (1, 835.3542232513428),   # Baseline
    (4, 916.5659439563751),    # 4 executors
]

table = create_speedup_table(example_measurements)
print("Speedup Analysis Table")
print("=" * 50)
print(table.to_string(index=False))

Speedup Analysis Table
 Executors Time (sec) Speedup Efficiency
         1      835.4   1.00x       100%
         4      916.6   0.91x        23%


## Model 2 - SPECIES

Preprocessing for second model. Currently markdown because not the main focus of the this milestone.

Differs from model 1 by separating by species instead of health.

In [ ]:
# FOR MODEL 2 - SPECIES
# # We want 70% of all plant samples to go into our training data
# 15% of healthy/diseased samples should go into validation, and 15% into test sets.

# w_s = Window.partitionBy("plant").orderBy(F.rand(seed=1))

# metadata_s = metadata.withColumn("row_num", F.row_number().over(w_s))\
#            .withColumn("count", F.count("*").over(Window.partitionBy("plant")))

# metadata_s = metadata_s.withColumn("split", F.when(F.col("row_num") <= 0.7* F.col("count"), "train")\
#            .when(F.col("row_num") <= 0.85 * F.col("count"), "val").otherwise("test"))

# train_s = metadata_s.filter(F.col("split") == "train").select("file_path", "plant")
# val_s = metadata_s.filter(F.col("split") == "val").select("file_path", "plant")
# test_s = metadata_s.filter(F.col("split") == "test").select("file_path", "plant")

In [11]:
# train_s.printSchema()

root
 |-- file_path: string (nullable = true)
 |-- plant: string (nullable = true)



In [9]:
metadata.select("plant").distinct().show(40)

+------------+
|       plant|
+------------+
|       Wheat|
|        Rice|
|     Cabbage|
| Cauliflower|
|Bottle_gourd|
|    Cucumber|
|       Onion|
|    Capsicum|
|       Chili|
|      Tomato|
|      Potato|
|        Corn|
|         Pea|
+------------+



In [ ]:
# Check counts - SPECIES
# print(train_s.count())
# print(val_s.count())
# print(test_s.count())

In [ ]:
# train_s.filter("plant = 'Chili'").take(5)

Label encoding for multiclass (species) classification

In [15]:
# Create label encoder and fit to the training set 
# indexer_s = StringIndexer(inputCol="plant", outputCol="label") 
# le_s = indexer_s.fit(train_s)

# Label encode all sets
# train_s = le_s.transform(train_s) 
# val_s = le_s.transform(val_s) 
# test_s = le_s.transform(test_s) 

In [ ]:
# print(train_s.take(3))
# print(val_s.take(3))
# print(test_s.take(3))
# print(train_s.count())

In [ ]:
# Add a weights column for the weightcol parameter to tune the models because of our data imbalance
# ONLY do this for training data
# train_s.cache()

# class_counts_s = train_s.groupBy("label").count()

# total_s = train_s.count()
# num_classes_s = class_counts_s.count()

# set a max weight because without it, some classes are weighted like 8 and some 0.22 
# max_weight = 5.0

# weights_s = class_counts_s.withColumn(
#    "weight",
#    F.lit(train_s.count()) / (F.lit(class_counts_s.count()) * F.col("count"))
#).select("label", "weight")

# train_s = train_s.join(weights_s, on="label", how="left")

In [ ]:
# train_s.groupBy("label").agg(F.first("weight")).show()

Preprocessing images

In [ ]:
# Join the images to the data
# Doing this now because it was giving me memory problems when I brought in the images earlier
# with all of the other transformations I did

#train_s = train_s.join(images, on = "file_path", how = "inner")
#val_s = val_s.join(images, on = "file_path", how = "inner")
#test_s = test_s.join(images, on = "file_path", how = "inner")

In [21]:
#train_s.printSchema()

root
 |-- file_path: string (nullable = true)
 |-- label: double (nullable = false)
 |-- plant: string (nullable = true)
 |-- weight: double (nullable = true)
 |-- content: binary (nullable = true)



In [ ]:
#train_s = train_s.withColumn("features_array", decode_image("content")) \
#        .filter(F.col("features_array").isNotNull()) \
#        .withColumn("features", array_to_vector("features_array"))\
#        .select("file_path", "label", "weight", "features")

#val_s = val_s.withColumn("features_array", decode_image("content")) \
#        .filter(F.col("features_array").isNotNull()) \
#        .withColumn("features", array_to_vector("features_array"))\
#        .select("file_path", "label",  "features")
#test_s = test_s.withColumn("features_array", decode_image("content")) \
#        .filter(F.col("features_array").isNotNull()) \
#        .withColumn("features", array_to_vector("features_array"))\
#        .select("file_path", "label", "features")

In [24]:
# check decoding went ok. took 5 min on 4 cores 128 gb
# val_s.filter(F.col("features").isNotNull()).count()

7855

In [ ]:
# # Create a new column, pixels, from the content column that is in the correct format for the models
# # Then drop the columns we don't need
# # Drop any nulls where decoding failed

# train_s = train_s.withColumn("features", decode_udf("content")) \
#     .select("file_path", "label", "features", "weight").filter(F.col("features").isNotNull())

# # # Drop any nulls where decoding failed
# # train_s= train_s.filter(F.col("features").isNotNull())

In [ ]:
# sample_check = train_s.limit(100)

# sample_check.filter(F.col("features").isNotNull()).count()

# print(train_s.count())

In [ ]:
# train_s.printSchema()

In [ ]:
# # # Check dimension consistency across the data
# # train_s.select(
# #     F.size(vector_to_array("features")).alias("dim")
# # ).limit(10).show()

# # # Check for nulls
# # train_s.select("features").where(F.col("features").isNull()).limit(10).show()
# train_s = train_s.filter(F.col("features").isNotNull())

In [ ]:
# Do the same for the rest of the data sets

In [ ]:
# val_s = val_s.withColumn("features", decode_udf("content")) \
#     .select("file_path", "label", "features","weight").filter(F.col("features").isNotNull())

# test_s = test_s.withColumn("features", decode_udf("content")) \
#     .select("file_path", "label", "features","weight").filter(F.col("features").isNotNull())

# train_d = train_d.withColumn("features", decode_udf("content")) \
#     .select("file_path", "label", "features","weight").filter(F.col("features").isNotNull())

# val_d = val_d.withColumn("features", decode_udf("content")) \
#     .select("file_path", "label", "features","weight").filter(F.col("features").isNotNull())

# test_d = test_d.withColumn("features", decode_udf("content")) \
#     .select("file_path", "label", "features","weight").filter(F.col("features").isNotNull())

In [ ]:
# val_s.filter(F.col("features").isNotNull()).count()